In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# -----------------------
# One N-HiTS Block
# -----------------------
class NHiTSBlock(nn.Module):
    def __init__(self, input_size, forecast_size, hidden_size=256, downsample=1):
        super().__init__()
        self.downsample = downsample
        reduced_size = input_size // downsample

        # Fully connected layers
        self.fc1 = nn.Linear(reduced_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, forecast_size // downsample)

    def forward(self, x):
        # Downsample input
        if self.downsample > 1:
            x = x[:, ::self.downsample]   # take every k-th point

        # MLP
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        forecast_coarse = self.fc3(h)

        # Interpolate back to full forecast horizon
        forecast = F.interpolate(
            forecast_coarse.unsqueeze(1),
            size=x.size(1) * self.downsample,  # upscale back
            mode="linear",
            align_corners=False
        ).squeeze(1)

        return forecast

In [ ]:
# -----------------------
# N-HiTS Model
# -----------------------
class NHiTS(nn.Module):
    def __init__(self, input_size, forecast_size, hidden_size=256, n_blocks=3):
        super().__init__()
        self.blocks = nn.ModuleList([
            NHiTSBlock(input_size, forecast_size, hidden_size, downsample=2**i)
            for i in range(n_blocks)
        ])

    def forward(self, x):
        forecast = 0
        for block in self.blocks:
            forecast_block = block(x)
            forecast = forecast + forecast_block  # add refinements
        return forecast


### Now code for test

In [ ]:
# -----------------------
# Example Usage
# -----------------------
batch_size = 16
input_size = 100   # past 100 timesteps
forecast_size = 20 # predict next 20 timesteps

model = NHiTS(input_size, forecast_size, hidden_size=128, n_blocks=3)
x = torch.randn(batch_size, input_size)
y_pred = model(x)

print("Input:", x.shape)
print("Forecast:", y_pred.shape)

### Now code for train

In [ ]:

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(10):
    optimizer.zero_grad()
    y_pred = model(x)
    loss = criterion(y_pred, y_true)   # y_true = actual future sequence
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
